<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Temperature_Celsius/Temperature_Hybrid_Approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install river --quiet

In [ ]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [ ]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/final_temperature_dataset.csv"
temp_df = pd.read_csv(path)

print("Dataset shape:", temp_df.shape)
temp_df.head()

Dataset shape: (130003, 16)


,uv_index,latitude,humidity,pressure_mb,air_quality_Ozone,condition_text,air_quality_Nitrogen_dioxide,cloud,visibility_km,longitude,air_quality_PM10,wind_mph,gust_mph,wind_degree,precip_mm,temperature_celsius
0,1.0,46.60,58,1012.0,62.2,2,2.5,0,16.0,-120.49,7.1,4.3,10.3,220,0.00,16.1
1,1.0,14.10,78,1017.0,23.3,32,3.7,37,10.0,-87.22,25.3,3.8,7.0,240,0.28,23.0
2,1.0,13.71,94,1010.0,5.9,23,7.7,50,10.0,-89.20,28.1,2.2,2.8,182,0.30,26.0
3,1.0,14.62,88,1019.0,0.4,19,35.0,100,5.0,-90.53,178.1,13.6,18.1,190,0.09,20.0
4,1.0,17.25,89,1007.0,34.0,30,0.3,94,10.0,-88.77,32.1,4.3,6.5,99,0.00,26.0


In [ ]:
target = "temperature_celsius"

In [ ]:


# -----------------------------
# TEMPERATURE LAG FEATURES
# -----------------------------
temp_df["temp_lag1"] = temp_df[target].shift(1)
temp_df["temp_lag2"] = temp_df[target].shift(2)
temp_df["temp_lag3"] = temp_df[target].shift(3)
temp_df["temp_lag5"] = temp_df[target].shift(5)
temp_df["temp_lag7"] = temp_df[target].shift(7)

# -----------------------------
# ROLLING FEATURES FOR TEMPERATURE
# -----------------------------
temp_df["temp_roll3_mean"] = temp_df[target].rolling(window=3).mean()
temp_df["temp_roll5_mean"] = temp_df[target].rolling(window=5).mean()
temp_df["temp_roll7_mean"] = temp_df[target].rolling(window=7).mean()

temp_df["temp_roll3_std"] = temp_df[target].rolling(window=3).std()
temp_df["temp_roll5_std"] = temp_df[target].rolling(window=5).std()

# -----------------------------
# RELATED WEATHER FEATURE LAGS
# -----------------------------
if "humidity" in temp_df.columns:
    temp_df["humidity_lag1"] = temp_df["humidity"].shift(1)
    temp_df["humidity_lag3"] = temp_df["humidity"].shift(3)

if "pressure_mb" in temp_df.columns:
    temp_df["pressure_lag1"] = temp_df["pressure_mb"].shift(1)
    temp_df["pressure_lag3"] = temp_df["pressure_mb"].shift(3)

if "wind_mph" in temp_df.columns:
    temp_df["wind_lag1"] = temp_df["wind_mph"].shift(1)
    temp_df["wind_lag3"] = temp_df["wind_mph"].shift(3)

if "gust_mph" in temp_df.columns:
    temp_df["gust_lag1"] = temp_df["gust_mph"].shift(1)

if "cloud" in temp_df.columns:
    temp_df["cloud_lag1"] = temp_df["cloud"].shift(1)

if "precip_mm" in temp_df.columns:
    temp_df["precip_lag1"] = temp_df["precip_mm"].shift(1)

# -----------------------------
# INTERACTION FEATURES
# -----------------------------
if "humidity" in temp_df.columns:
    temp_df["humidity_temp_interaction"] = temp_df["humidity"] * temp_df[target]

if "wind_mph" in temp_df.columns and "cloud" in temp_df.columns:
    temp_df["wind_cloud_interaction"] = temp_df["wind_mph"] * temp_df["cloud"]

if "pressure_mb" in temp_df.columns and "humidity" in temp_df.columns:
    temp_df["pressure_humidity_interaction"] = temp_df["pressure_mb"] * temp_df["humidity"]

# -----------------------------
# REMOVE NA CREATED BY LAGS
# -----------------------------
temp_df = temp_df.dropna().reset_index(drop=True)

print("After feature engineering:", temp_df.shape)
temp_df.head()

After feature engineering: (129989, 39)


,uv_index,latitude,humidity,pressure_mb,air_quality_Ozone,condition_text,air_quality_Nitrogen_dioxide,cloud,visibility_km,longitude,...,wind_lag1,wind_lag3,humidity_temp_interaction,wind_pm10_interaction,pressure_lag3,gust_lag1,cloud_lag1,precip_lag1,wind_cloud_interaction,pressure_humidity_interaction
0,1.0,17.30,79,1013.0,26.5,2,0.3,0,10.0,-62.72,...,12.5,7.4,2133.0,33.21,1013.0,14.3,75.0,0.06,0.0,80027.0
1,1.0,12.05,79,1011.0,20.2,32,0.8,25,10.0,-61.75,...,8.1,9.4,2212.0,163.20,1011.0,19.6,0.0,0.00,340.0,79869.0
2,1.0,6.80,100,1011.0,20.6,32,2.9,50,8.0,-58.17,...,13.6,12.5,2400.0,3.52,1012.0,18.1,25.0,0.06,110.0,101100.0
3,1.0,-2.08,98,1009.0,1.0,4,0.2,100,0.0,-58.17,...,2.2,8.1,2263.8,1.25,1013.0,6.6,50.0,0.08,250.0,98882.0
4,1.0,18.47,94,1014.0,0.5,32,14.4,50,10.0,-69.90,...,2.5,13.6,2162.0,102.06,1011.0,5.0,100.0,0.04,405.0,95316.0


In [ ]:
selected_features = [col for col in temp_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 38
['uv_index', 'latitude', 'humidity', 'pressure_mb', 'air_quality_Ozone', 'condition_text', 'air_quality_Nitrogen_dioxide', 'cloud', 'visibility_km', 'longitude', 'air_quality_PM10', 'wind_mph', 'gust_mph', 'wind_degree', 'precip_mm', 'temp_lag1', 'temp_lag2', 'temp_lag3', 'temp_lag5', 'temp_lag7', 'temp_roll3_mean', 'temp_roll5_mean', 'temp_roll7_mean', 'temp_roll3_std', 'temp_roll5_std', 'humidity_lag1', 'humidity_lag3', 'pressure_lag1', 'wind_lag1', 'wind_lag3', 'humidity_temp_interaction', 'wind_pm10_interaction', 'pressure_lag3', 'gust_lag1', 'cloud_lag1', 'precip_lag1', 'wind_cloud_interaction', 'pressure_humidity_interaction']


In [ ]:
split_index = int(0.8 * len(temp_df))

train_df = temp_df.iloc[:split_index].copy()
test_df = temp_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103991, 39)
Test shape : (25998, 39)


In [ ]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34317, 39)
Iteration 2 shape: (34317, 39)
Iteration 3 shape: (35357, 39)


In [ ]:
try:
    from river import forest

    river_model = forest.ARFRegressor(
        n_models=20,
        max_features="sqrt",
        lambda_value=6,
        seed=42
    )
    model_name = "River ARF Hybrid"

except Exception:
    from river import tree

    river_model = compose.Pipeline(
        preprocessing.StandardScaler(),
        tree.HoeffdingAdaptiveTreeRegressor(
            grace_period=50,
            delta=1e-5,
            leaf_prediction="adaptive"
        )
    )
    model_name = "River HAT Hybrid"

print("Using model:", model_name)

Using model: River ARF Hybrid


In [ ]:
def run_hybrid_iteration(data, model, iteration_name, selected_features, target, warmup=False):
    mse = metrics.MSE()
    rmse = metrics.RMSE()
    mae = metrics.MAE()
    r2 = metrics.R2()

    y_true_all = []
    y_pred_all = []

    first_part = int(0.1 * len(data)) if warmup else 0

    for i, (_, row) in enumerate(data.iterrows()):
        x = row[selected_features].to_dict()
        y = row[target]

        if i < first_part:
            model.learn_one(x, y)
            continue

        y_pred = model.predict_one(x)
        if y_pred is None:
            y_pred = 0.0

        y_true_all.append(y)
        y_pred_all.append(y_pred)

        mse.update(y, y_pred)
        rmse.update(y, y_pred)
        mae.update(y, y_pred)
        r2.update(y, y_pred)

        model.learn_one(x, y)

    print(f"\n{iteration_name}")
    print("MSE :", round(mse.get(), 4))
    print("RMSE:", round(rmse.get(), 4))
    print("MAE :", round(mae.get(), 4))
    print("R2  :", round(r2.get(), 4))
    print("Accuracy (%):", round(r2.get() * 100, 2))

    return model, mse.get(), rmse.get(), mae.get(), r2.get(), y_true_all, y_pred_all

In [ ]:
river_results = []

river_model, mse1, rmse1, mae1, r21, y_true1, y_pred1 = run_hybrid_iteration(
    iter1, river_model, "Iteration 1", selected_features, target, warmup=True
)

river_results.append([
    "Iteration 1", model_name, mse1, rmse1, mae1, r21, r21 * 100
])


Iteration 1
MSE : 8.179
RMSE: 2.8599
MAE : 2.164
R2  : 0.8529
Accuracy (%): 85.29


In [ ]:
river_model, mse2, rmse2, mae2, r22, y_true2, y_pred2 = run_hybrid_iteration(
    iter2, river_model, "Iteration 2", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 2", model_name, mse2, rmse2, mae2, r22, r22 * 100
])


Iteration 2
MSE : 11.5559
RMSE: 3.3994
MAE : 2.3897
R2  : 0.8904
Accuracy (%): 89.04


In [ ]:
river_model, mse3, rmse3, mae3, r23, y_true3, y_pred3 = run_hybrid_iteration(
    iter3, river_model, "Iteration 3", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 3", model_name, mse3, rmse3, mae3, r23, r23 * 100
])


Iteration 3
MSE : 6.3935
RMSE: 2.5285
MAE : 1.903
R2  : 0.8824
Accuracy (%): 88.24


In [ ]:
river_results_df = pd.DataFrame(
    river_results,
    columns=["Iteration", "Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

river_results_df

,Iteration,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Iteration 1,River ARF Hybrid,8.179,2.860,2.164,0.853,85.289
1,Iteration 2,River ARF Hybrid,11.556,3.399,2.390,0.890,89.041
2,Iteration 3,River ARF Hybrid,6.393,2.529,1.903,0.882,88.244


In [ ]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = river_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nFinal Test Performance")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Final Test Performance
MSE : 15.2228
RMSE: 3.9016
MAE : 2.5762
R2  : 0.8774
Accuracy (%): 87.74


In [ ]:
river_final_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

river_final_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River ARF Hybrid,15.223,3.902,2.576,0.877,87.744


In [ ]:
from google.colab import files

river_results_df.to_csv("river_hybrid_temp_iteration_results.csv", index=False)
river_final_test_df.to_csv("river_hybrid_temp_final_test_results.csv", index=False)

files.download("river_hybrid_temp_iteration_results.csv")
files.download("river_hybrid_temp_final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>